In [3]:
import gseapy as gp

# List available libraries
libs = gp.get_library_name()

print([x for x in libs if "Hallmark" in x or "KEGG" in x])

['KEGG_2013', 'KEGG_2015', 'KEGG_2016', 'KEGG_2019_Human', 'KEGG_2019_Mouse', 'KEGG_2021_Human', 'KEGG_2026', 'MSigDB_Hallmark_2020']


In [4]:
import requests

hallmark_name = "MSigDB_Hallmark_2020" if "MSigDB_Hallmark_2020" in libs else next(
    x for x in libs if "Hallmark" in x
)
try:
    hallmark = gp.get_library(name=hallmark_name, organism="Human")
except TypeError as e:
    if "bytes-like object is required" not in str(e):
        raise


    url = f"https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName={hallmark_name}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()

    hallmark = {}
    for line in r.text.strip().splitlines():
        term, *genes = line.split("\t")
        hallmark[term] = genes

print(len(hallmark))  # ~50 pathways
target = "HALLMARK_DNA_REPAIR"

def _norm(s):
    return s.lower().replace("hallmark_", "").replace(" ", "_").replace("-", "_").replace("/", "_")

match = target if target in hallmark else next((k for k in hallmark if _norm(k) == _norm(target)), None)
if match is None:
    raise KeyError(f"{target} not found. Example keys: {list(hallmark)[:5]}")

print([g for g in hallmark[match] if g][:10])

50
['POLR2H', 'POLR2A', 'POLR2G', 'POLR2E', 'POLR2J', 'POLR2F', 'POLR2C', 'POLR2K', 'GTF2H3', 'POLR2D']


In [5]:
# Summarize what was pulled
if "hallmark" not in globals() or not isinstance(hallmark, dict):
    raise RuntimeError("'hallmark' is not in memory. Run the previous cell first.")

import json
from pathlib import Path
import pandas as pd

# Build per-set stats table
df_sets = pd.DataFrame(
    {
        "set_name": list(hallmark.keys()),
        "n_genes": [len([g for g in genes if g]) for genes in hallmark.values()],
    }
).sort_values("n_genes", ascending=False).reset_index(drop=True)

print(f"Total gene sets pulled: {len(df_sets):,}")
print(f"Total unique genes across all sets: {len({g for genes in hallmark.values() for g in genes if g}):,}")
print("Genes per set summary:")
print(df_sets["n_genes"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_string())

print("\nTop 10 largest sets:")
print(df_sets.head(10).to_string(index=False))

print("\nTop 10 smallest sets:")
print(df_sets.tail(10).sort_values("n_genes").to_string(index=False))

# Save for easy reading/reuse
out_dir = Path("./results/gene_sets")
out_dir.mkdir(parents=True, exist_ok=True)

summary_csv = out_dir / "hallmark_set_sizes.csv"
summary_md = out_dir / "hallmark_set_sizes.md"
gene_sets_json = out_dir / "hallmark_gene_sets.json"

df_sets.to_csv(summary_csv, index=False)
summary_md.write_text(df_sets.to_markdown(index=False), encoding="utf-8")
gene_sets_json.write_text(json.dumps(hallmark, indent=2), encoding="utf-8")

print("\nSaved files:")
print(f"- {summary_csv}")
print(f"- {summary_md}")
print(f"- {gene_sets_json}")

Total gene sets pulled: 50
Total unique genes across all sets: 4,383
Genes per set summary:
count     50.000000
mean     146.420000
std       62.275913
min       32.000000
25%       97.750000
50%      180.000000
75%      200.000000
90%      200.000000
95%      200.000000
max      200.000000

Top 10 largest sets:
                     set_name  n_genes
TNF-alpha Signaling via NF-kB      200
                      Hypoxia      200
              G2-M Checkpoint      200
       Estrogen Response Late      200
      Estrogen Response Early      200
                 Adipogenesis      200
                   Myogenesis      200
            KRAS Signaling Dn      200
        Xenobiotic Metabolism      200
    Oxidative Phosphorylation      200

Top 10 smallest sets:
                       set_name  n_genes
                Notch Signaling       32
             Hedgehog Signaling       36
                   Angiogenesis       36
            Pancreas Beta Cells       40
     Wnt-beta Catenin Signali

## Pathway Scoring Head on Frozen RNA Foundation Model

This section builds an interpretable pathway-scoring head on top of the pretrained ExpressionPerformer.

### Goal
- Freeze pretrained backbone.
- Train a small head to map contextualized gene embeddings to Hallmark pathway logits.
- Produce:
  1. Gene-level pathway scores (`genes x pathways`)
  2. Sample-level pathway scores
  3. Condition-difference scores (for example, flight minus ground)
  4. Top responsible genes per pathway/condition

### Data Shapes
- Backbone hidden states: `[batch, genes, hidden_dim]`
- Pathway labels per gene: `[genes, pathways]`
- Head logits per sample: `[batch, genes, pathways]`


In [6]:
import argparse
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from generate_archs4_embeddings import ExpressionPerformer, _strip_module_prefix

# -------------------------
# Configuration
# -------------------------
CFG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
    "data_dir": Path("./data/archs4/train_orthologs_sharded"),
    "checkpoint": Path("./checkpoints_performer/r7hnr92k/best_model.pt"),
    "canonical_genes_csv": Path("./data/archs4/train_orthologs/canonical_genes.csv"),
    "out_dir": Path("./results/pathway_head"),
    "subset_shards": 4,
    "max_samples_per_shard": 128,
    "batch_size": 2,
    "epochs": 2,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "hidden_mlp": 256,
}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

CFG["out_dir"].mkdir(parents=True, exist_ok=True)
print("Using device:", CFG["device"])


Using device: cuda


In [7]:
# -------------------------
# Build pathway label matrix: [genes, pathways]
# -------------------------
if "hallmark" not in globals() or not isinstance(hallmark, dict):
    hallmark_path = Path("./results/gene_sets/hallmark_gene_sets.json")
    if not hallmark_path.exists():
        raise FileNotFoundError("Hallmark sets not found in memory or on disk. Run earlier cells first.")
    hallmark = json.loads(hallmark_path.read_text())

canonical_genes = pd.read_csv(CFG["canonical_genes_csv"])["gene_symbol"].astype(str).tolist()
pathway_names = list(hallmark.keys())

gene_to_idx = {g: i for i, g in enumerate(canonical_genes)}
Y = np.zeros((len(canonical_genes), len(pathway_names)), dtype=np.float32)

for p_idx, p_name in enumerate(pathway_names):
    for g in hallmark[p_name]:
        if g in gene_to_idx:
            Y[gene_to_idx[g], p_idx] = 1.0

Y_tensor = torch.from_numpy(Y)

print("Genes:", len(canonical_genes))
print("Pathways:", len(pathway_names))
print("Label matrix shape:", tuple(Y_tensor.shape))
print("Avg pathways per gene:", float(Y_tensor.sum(1).mean()))
print("Avg genes per pathway:", float(Y_tensor.sum(0).mean()))


Genes: 15165
Pathways: 50
Label matrix shape: (15165, 50)
Avg pathways per gene: 0.43837785720825195
Avg genes per pathway: 132.9600067138672


In [8]:
# -------------------------
# Load frozen backbone and define pathway head
# -------------------------
ckpt = torch.load(CFG["checkpoint"], map_location="cpu")
ccfg = dict(ckpt.get("config", {}))

backbone = ExpressionPerformer(
    num_genes=len(canonical_genes),
    hidden_dim=int(ccfg.get("hidden_dim", 512)),
    n_heads=int(ccfg.get("num_heads", 8)),
    n_layers=int(ccfg.get("num_layers", 4)),
    ffn_dim=int(ccfg.get("ffn_dim", int(ccfg.get("hidden_dim", 512)) * 4)),
    ree_base=float(ccfg.get("ree_base", 100.0)),
    mask_token_id=float(ccfg.get("mask_token", -10.0)),
    feature_type=str(ccfg.get("feature_type", "sqr")),
    compute_type=str(ccfg.get("compute_type", "iter")),
    include_species_embedding=bool(ccfg.get("include_species_embedding", False)),
    num_species=2,
)
state = _strip_module_prefix(ckpt["model_state_dict"])
_ = backbone.load_state_dict(state, strict=False)
backbone = backbone.to(CFG["device"]).eval()
for p in backbone.parameters():
    p.requires_grad = False

class PathwayHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, n_pathways: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, n_pathways),
        )

    def forward(self, h):
        # h: [B, G, D] -> logits: [B, G, P]
        return self.net(h)

head = PathwayHead(
    in_dim=int(ccfg.get("hidden_dim", 512)),
    hidden_dim=int(CFG["hidden_mlp"]),
    n_pathways=len(pathway_names),
).to(CFG["device"])

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(head.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

print("Backbone hidden dim:", int(ccfg.get("hidden_dim", 512)))
print("Pathway head output dim:", len(pathway_names))


Backbone hidden dim: 512
Pathway head output dim: 50


In [9]:
# -------------------------
# ARCHS4 subset loader
# -------------------------
class Archs4SubsetDataset(Dataset):
    def __init__(self, data_dir: Path, gene_cols: list[str], subset_shards: int, max_samples_per_shard: int):
        batch_dir = data_dir / "batch_files"
        if not batch_dir.exists():
            batch_dir = data_dir
        shard_paths = sorted(batch_dir.glob("*.parquet"))[:subset_shards]
        if not shard_paths:
            raise FileNotFoundError(f"No parquet shards found in {batch_dir}")

        rows = []
        for sp in shard_paths:
            tbl = pq.read_table(sp, columns=gene_cols + ["geo_accession"])
            df = tbl.to_pandas()
            if len(df) > max_samples_per_shard:
                df = df.sample(max_samples_per_shard, random_state=CFG["seed"])
            rows.append(df)

        all_df = pd.concat(rows, ignore_index=True)
        self.geo = all_df["geo_accession"].astype(str).tolist()
        self.x = all_df[gene_cols].to_numpy(dtype=np.float32)

        norm = str(ccfg.get("normalization", "log1p_tpm"))
        if norm == "log1p_tpm":
            self.x = np.log1p(np.maximum(self.x, 0.0)).astype(np.float32, copy=False)

    def __len__(self):
        return len(self.geo)

    def __getitem__(self, idx):
        return torch.from_numpy(self.x[idx]), self.geo[idx]


try:
    dataset = Archs4SubsetDataset(
        data_dir=CFG["data_dir"],
        gene_cols=canonical_genes,
        subset_shards=int(CFG["subset_shards"]),
        max_samples_per_shard=int(CFG["max_samples_per_shard"]),
    )
except KeyError as e:
    if "geo_accession" not in str(e):
        raise

    batch_dir = CFG["data_dir"] / "batch_files"
    if not batch_dir.exists():
        batch_dir = CFG["data_dir"]
    shard_paths = sorted(batch_dir.glob("*.parquet"))[: int(CFG["subset_shards"])]
    if not shard_paths:
        raise FileNotFoundError(f"No parquet shards found in {batch_dir}")

    rows = []
    id_candidates = ["geo_accession", "sample_id", "sample", "gsm", "accession"]

    for sp in shard_paths:
        schema_cols = set(pq.read_schema(sp).names)
        id_col = next((c for c in id_candidates if c in schema_cols), None)
        present_genes = [g for g in canonical_genes if g in schema_cols]
        if not present_genes:
            continue

        read_cols = present_genes + ([id_col] if id_col else [])
        df = pq.read_table(sp, columns=read_cols).to_pandas()

        if len(df) > int(CFG["max_samples_per_shard"]):
            df = df.sample(int(CFG["max_samples_per_shard"]), random_state=CFG["seed"])

        # Fill missing canonical genes with 0
        for g in canonical_genes:
            if g not in df.columns:
                df[g] = 0.0

        if id_col and id_col in df.columns:
            df["geo_accession"] = df[id_col].astype(str)
        else:
            df["geo_accession"] = [f"{sp.stem}_{i}" for i in range(len(df))]

        rows.append(df[canonical_genes + ["geo_accession"]])

    if not rows:
        raise RuntimeError("No usable shard rows found after schema filtering.")

    all_df = pd.concat(rows, ignore_index=True)
    x_np = all_df[canonical_genes].to_numpy(dtype=np.float32)

    if str(ccfg.get("normalization", "log1p_tpm")) == "log1p_tpm":
        x_np = np.log1p(np.maximum(x_np, 0.0)).astype(np.float32, copy=False)

    # Training loop only uses xb, so second tensor is a dummy label.
    dataset = torch.utils.data.TensorDataset(
        torch.from_numpy(x_np),
        torch.zeros(len(x_np), dtype=torch.long),
    )
loader = DataLoader(dataset, batch_size=int(CFG["batch_size"]), shuffle=True, drop_last=False)

print("Training subset samples:", len(dataset))


Training subset samples: 512


In [10]:
# -------------------------
# Train pathway head on frozen embeddings
# -------------------------
head.train()
loss_history = []

Y_dev = Y_tensor.to(CFG["device"])  # [G, P]

for epoch in range(int(CFG["epochs"])):
    epoch_losses = []
    for xb, _ in loader:
        xb = xb.to(CFG["device"])  # [B, G]

        with torch.no_grad():
            h = backbone._encode_hidden(xb, None)  # [B, G, D]

        logits = head(h)  # [B, G, P]
        target = Y_dev.unsqueeze(0).expand(logits.size(0), -1, -1)  # [B, G, P]

        loss = criterion(logits, target)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_losses.append(float(loss.item()))

    mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    loss_history.append(mean_loss)
    print(f"Epoch {epoch+1}/{CFG['epochs']} - loss: {mean_loss:.6f}")

pd.DataFrame({"epoch": np.arange(1, len(loss_history)+1), "loss": loss_history})


Epoch 1/2 - loss: 0.061354
Epoch 2/2 - loss: 0.038440


,epoch,loss
0,1,0.061354
1,2,0.038440


In [11]:
# -------------------------
# Score OSDR samples and compute condition-level pathway deltas
# -------------------------
from demo_osdr_top5 import load_random_osdr_sample_vector
import argparse  # ensure available if cell run in isolation
import hashlib

# Build Y_df: membership matrix [genes x pathways] with 1/0 labels
Y_df = pd.DataFrame(Y, index=canonical_genes, columns=pathway_names).astype(int)


def membership_sample_score(gene_level: pd.DataFrame, Y_df: pd.DataFrame, top_k: int = 10) -> pd.DataFrame:
    """
    For each pathway, compute two scores:
      - mean_score : mean predicted prob across ALL member genes
      - topk_score : mean of the top-k member-gene scores
    Returns a DataFrame indexed by pathway with columns [mean_score, topk_score].
    """
    rows = []
    for pw in gene_level.columns:
        members = Y_df.index[Y_df[pw] == 1]
        if len(members) == 0:
            rows.append({"pathway": pw, "mean_score": float("nan"), "topk_score": float("nan")})
            continue
        scores = gene_level.loc[members, pw]
        mean_sc = float(scores.mean())
        topk_sc = float(scores.nlargest(min(top_k, len(scores))).mean())
        rows.append({"pathway": pw, "mean_score": mean_sc, "topk_score": topk_sc})
    return pd.DataFrame(rows).set_index("pathway")


def _cache_key(sample_name: str) -> str:
    return hashlib.sha1(sample_name.encode("utf-8")).hexdigest()[:16]


def score_single_osdr_sample(
    sample_name: str,
    top_k: int = 10,
    use_disk_cache: bool = True,
    cache_dir: Path | None = None,
    return_cache_status: bool = False,
):
    if cache_dir is None:
        cache_dir = CFG["out_dir"] / "osdr_embedding_cache"
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    sample_key = _cache_key(sample_name)
    cache_npz = cache_dir / f"{sample_key}.npz"
    cache_meta = cache_dir / f"{sample_key}.json"

    # Fast path: load cached outputs and skip model forward pass.
    if use_disk_cache and cache_npz.exists() and cache_meta.exists():
        arr = np.load(cache_npz)
        probs = arr["probs"].astype(np.float32, copy=False)
        sample_embedding = arr["sample_embedding"].astype(np.float32, copy=False)
        meta = json.loads(cache_meta.read_text(encoding="utf-8"))
        sample_id = str(meta.get("sample_id", sample_name))

        gene_level = pd.DataFrame(probs, index=canonical_genes, columns=pathway_names)
        sample_scores = membership_sample_score(gene_level, Y_df, top_k=top_k)
        if return_cache_status:
            return sample_id, None, gene_level, sample_scores, True, sample_embedding
        return sample_id, None, gene_level, sample_scores

    local_args = argparse.Namespace(**CFG)
    local_args.osdr_sample_name = sample_name
    local_args.osdr_data_dir = Path("./data/osdr")
    local_args.osdr_metadata = None
    local_args.orthologs = Path("./data/ensembl/orthologs_one2one.txt")
    local_args.canonical_genes = CFG["canonical_genes_csv"]
    local_args.mouse_exon_lengths = Path("./data/gencode/gencode_v49_mouse_gene_exon_lengths.csv")

    x, sample_id, row = load_random_osdr_sample_vector(local_args)

    xb = torch.from_numpy(x[None, :]).to(CFG["device"])
    with torch.no_grad():
        h = backbone._encode_hidden(xb, None)
        logits = head(h)
        probs = torch.sigmoid(logits)[0].detach().cpu().numpy()  # [G, P]
        sample_embedding = h[0].mean(dim=0).detach().cpu().numpy()  # [D]

    # gene_level: full [genes x pathways] probability matrix
    gene_level = pd.DataFrame(probs, index=canonical_genes, columns=pathway_names)
    # sample_level: membership-restricted scores [pathways x {mean_score, topk_score}]
    sample_scores = membership_sample_score(gene_level, Y_df, top_k=top_k)

    if use_disk_cache:
        np.savez_compressed(
            cache_npz,
            probs=probs.astype(np.float16),
            sample_embedding=sample_embedding.astype(np.float16),
        )
        cache_meta.write_text(
            json.dumps(
                {
                    "sample_name": sample_name,
                    "sample_id": str(sample_id),
                    "cache_key": sample_key,
                    "n_genes": int(probs.shape[0]),
                    "n_pathways": int(probs.shape[1]),
                    "embedding_dim": int(sample_embedding.shape[0]),
                },
                indent=2,
            ),
            encoding="utf-8",
        )

    if return_cache_status:
        return sample_id, row, gene_level, sample_scores, False, sample_embedding
    return sample_id, row, gene_level, sample_scores


# ---- OSD-48 liver: C57BL/6J, Female, 37-day mission ----
# Cage-cohort (C) replicates only; same housing as GC counterparts.
liver_flt_samples = [
    "Mmus_C57-6J_LVR_FLT_C_Rep1_M25",
    "Mmus_C57-6J_LVR_FLT_C_Rep2_M26",
    "Mmus_C57-6J_LVR_FLT_C_Rep3_M27",
    "Mmus_C57-6J_LVR_FLT_C_Rep4_M28",
    "Mmus_C57-6J_LVR_FLT_C_Rep5_M30",
]
liver_gnd_samples = [
    "Mmus_C57-6J_LVR_GC_C_Rep1_M36",
    "Mmus_C57-6J_LVR_GC_C_Rep2_M37",
    "Mmus_C57-6J_LVR_GC_C_Rep3_M38",
    "Mmus_C57-6J_LVR_GC_C_Rep4_M39",
    "Mmus_C57-6J_LVR_GC_C_Rep5_M40",
]

import argparse  # ensure available if cell run in isolation

print("Scoring flight samples...")
flt_sample_scores = []
flt_gene_prs = []
for name in liver_flt_samples:
    sid, meta, gene_lvl, sample_lvl = score_single_osdr_sample(name)
    flt_sample_scores.append(sample_lvl)
    flt_gene_prs.append(gene_lvl)
    print(f"  {sid}: top pathway (mean_score) = {sample_lvl['mean_score'].idxmax()}")

print("Scoring ground control samples...")
gnd_sample_scores = []
gnd_gene_prs = []
for name in liver_gnd_samples:
    sid, meta, gene_lvl, sample_lvl = score_single_osdr_sample(name)
    gnd_sample_scores.append(sample_lvl)
    gnd_gene_prs.append(gene_lvl)
    print(f"  {sid}: top pathway (mean_score) = {sample_lvl['mean_score'].idxmax()}")

# Average each score column across replicates
flt_mean_df = pd.concat(flt_sample_scores).groupby(level=0).mean()   # [pathways x {mean_score, topk_score}]
gnd_mean_df = pd.concat(gnd_sample_scores).groupby(level=0).mean()

# Deltas for both scoring methods
delta_mean  = (flt_mean_df["mean_score"]  - gnd_mean_df["mean_score"]).sort_values(ascending=False)
delta_topk  = (flt_mean_df["topk_score"]  - gnd_mean_df["topk_score"]).sort_values(ascending=False)

print("\n--- OSD-48 Liver: Top 10 flight-minus-ground pathways (membership mean) ---")
print(delta_mean.head(10).rename("delta_mean").to_string())
print("\n--- Bottom 10 (most suppressed in flight, membership mean) ---")
print(delta_mean.tail(10).rename("delta_mean").to_string())

# Top driver genes for the leading pathway
top_pw = delta_mean.index[0]
flt_gene_mean = pd.concat(flt_gene_prs).groupby(level=0).mean()
gnd_gene_mean = pd.concat(gnd_gene_prs).groupby(level=0).mean()
members_top = Y_df.index[Y_df[top_pw] == 1]
gene_delta = (
    (flt_gene_mean.loc[members_top, top_pw] - gnd_gene_mean.loc[members_top, top_pw])
    .sort_values(ascending=False)
)
print(f"\nTop 20 member genes driving [{top_pw}] in liver flight vs ground:")
print(gene_delta.head(20).to_string())

# Full results table: pathway, flight_mean, ground_mean, delta_mean, flight_topk, ground_topk, delta_topk
results_df = flt_mean_df.rename(columns={"mean_score": "flt_mean", "topk_score": "flt_topk"}
    ).join(gnd_mean_df.rename(columns={"mean_score": "gnd_mean", "topk_score": "gnd_topk"}))
results_df["delta_mean"] = results_df["flt_mean"] - results_df["gnd_mean"]
results_df["delta_topk"] = results_df["flt_topk"] - results_df["gnd_topk"]
results_df["n_members"] = [int(Y_df[pw].sum()) for pw in results_df.index]
results_df["study"] = "OSD-48"
results_df["tissue"] = "liver"
results_df = results_df.sort_values("delta_mean", ascending=False)
results_df


Scoring flight samples...
  OSD-48|Mmus_C57-6J_LVR_FLT_C_Rep1_M25: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
  OSD-48|Mmus_C57-6J_LVR_FLT_C_Rep2_M26: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
  OSD-48|Mmus_C57-6J_LVR_FLT_C_Rep3_M27: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
  OSD-48|Mmus_C57-6J_LVR_FLT_C_Rep4_M28: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
  OSD-48|Mmus_C57-6J_LVR_FLT_C_Rep5_M30: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
Scoring ground control samples...
  OSD-48|Mmus_C57-6J_LVR_GC_C_Rep1_M36: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
  OSD-48|Mmus_C57-6J_LVR_GC_C_Rep2_M37: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
  OSD-48|Mmus_C57-6J_LVR_GC_C_Rep3_M38: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
  OSD-48|Mmus_C57-6J_LVR_GC_C_Rep4_M39: top pathway (mean_score) = TNF-alpha Signaling via NF-kB
  OSD-48|Mmus_C57-6J_LVR_GC_C_Rep5_M40: top pathway (mean_scor

,flt_mean,flt_topk,gnd_mean,gnd_topk,delta_mean,delta_topk,n_members,study,tissue
pathway,,,,,,,,,
Mitotic Spindle,0.084610,0.444495,0.080078,0.408703,0.004532,0.035792,196,OSD-48,liver
Hypoxia,0.092925,0.481648,0.089492,0.456713,0.003433,0.024935,184,OSD-48,liver
Epithelial Mesenchymal Transition,0.242883,0.871977,0.239502,0.858635,0.003382,0.013343,188,OSD-48,liver
Adipogenesis,0.100400,0.504396,0.097255,0.474215,0.003145,0.030181,189,OSD-48,liver
Pancreas Beta Cells,0.017075,0.052885,0.014006,0.041637,0.003069,0.011248,39,OSD-48,liver
G2-M Checkpoint,0.188308,0.783715,0.185370,0.774572,0.002938,0.009143,184,OSD-48,liver
Complement,0.064281,0.322253,0.061815,0.281011,0.002466,0.041242,169,OSD-48,liver
Bile Acid Metabolism,0.070573,0.343753,0.068553,0.313038,0.002019,0.030715,106,OSD-48,liver
IL-2/STAT5 Signaling,0.052378,0.273862,0.051036,0.271391,0.001342,0.002471,180,OSD-48,liver


In [12]:
# -------------------------
# Save demo outputs for downstream review
# -------------------------
out_dir = CFG["out_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

# Save training loss
pd.DataFrame({"epoch": np.arange(1, len(loss_history)+1), "loss": loss_history}).to_csv(
    out_dir / "pathway_head_loss.csv", index=False
)

# Save condition-level summary
results_df.to_csv(out_dir / "osdr_condition_pathway_deltas.csv", index=False)

print("Saved:")
print("-", out_dir / "pathway_head_loss.csv")
print("-", out_dir / "osdr_condition_pathway_deltas.csv")
print("\nDemo objective:")
print("Input: OSDR RNA-seq sample -> Output: pathway summary + top responsible genes")


Saved:
- results/pathway_head/pathway_head_loss.csv
- results/pathway_head/osdr_condition_pathway_deltas.csv

Demo objective:
Input: OSDR RNA-seq sample -> Output: pathway summary + top responsible genes


In [13]:
# -------------------------
# All-accession condition analysis (Space Flight vs Ground Control)
# Statistical mechanics-inspired state analysis
# -------------------------
import time
import inspect

meta_path = Path("./data/osdr/metadata/selected_sample_metadata.tsv")
meta_df = pd.read_csv(meta_path, sep="\t")

acc_col = "id.accession"
sample_col = "id.sample name"
cond_col = "study.factor value.spaceflight"
tissue_col = "study.characteristics.material type"

required_cols = [acc_col, sample_col, cond_col, tissue_col]
missing = [c for c in required_cols if c not in meta_df.columns]
if missing:
    raise KeyError(f"Missing required metadata columns: {missing}")

meta_work = meta_df[required_cols].copy()
meta_work = meta_work.dropna(subset=[acc_col, sample_col, cond_col])
meta_work[acc_col] = meta_work[acc_col].astype(str).str.strip()
meta_work[sample_col] = meta_work[sample_col].astype(str).str.strip()
meta_work[cond_col] = meta_work[cond_col].astype(str).str.strip()
meta_work[tissue_col] = meta_work[tissue_col].fillna("unknown").astype(str).str.strip()

cond_norm = meta_work[cond_col].str.lower()
meta_work["condition_group"] = np.where(
    cond_norm.str.contains("space flight", na=False),
    "flight",
    np.where(cond_norm.str.contains("ground control", na=False), "ground", "other"),
)

eligible = meta_work[meta_work["condition_group"].isin(["flight", "ground"])].copy()

disk_cache_dir = CFG["out_dir"] / "osdr_embedding_cache"
disk_cache_dir.mkdir(parents=True, exist_ok=True)

sample_score_cache = {}
cache_stats = {"hit": 0, "miss": 0}
failed_samples = []
failed_reason_counts: dict[str, int] = {}
normalized_scores_by_cohort = {}

group_cols = [acc_col, tissue_col]
groups = list(eligible.groupby(group_cols, dropna=False))
total_groups = len(groups)
start_time = time.time()


def heartbeat(message: str) -> None:
    elapsed_min = (time.time() - start_time) / 60.0
    print(f"[heartbeat +{elapsed_min:.1f}m] {message}", flush=True)


def sample_score_frames_to_matrix(score_frames: list[pd.DataFrame], value_col: str = "mean_score") -> pd.DataFrame:
    if not score_frames:
        return pd.DataFrame(columns=pathway_names, dtype=np.float32)

    rows = []
    for i, sf in enumerate(score_frames):
        s = sf[value_col].astype(np.float32).reindex(pathway_names).fillna(0.0)
        s.name = f"sample_{i}"
        rows.append(s)

    out = pd.concat(rows, axis=1).T
    out = out.reindex(columns=pathway_names).fillna(0.0)
    return out.astype(np.float32)


def normalize_pathway_occupancies(sample_scores: pd.DataFrame, eps: float = 1e-12) -> pd.DataFrame:
    denom = sample_scores.sum(axis=1).astype(np.float64) + eps
    return sample_scores.div(denom, axis=0).astype(np.float32)


def compute_entropy_per_sample(occupancies: pd.DataFrame, eps: float = 1e-8) -> pd.DataFrame:
    p = occupancies.to_numpy(dtype=np.float64)
    h = -(p * np.log(p + eps)).sum(axis=1)
    n_pathways = p.shape[1]
    if n_pathways > 1:
        h_norm = h / np.log(float(n_pathways))
    else:
        h_norm = np.zeros_like(h)
    return pd.DataFrame({"H": h, "H_norm": h_norm}, index=occupancies.index)


def bootstrap_delta_ci(
    flt_occ: pd.DataFrame,
    gnd_occ: pd.DataFrame,
    n_bootstrap: int = 1000,
    random_state: int = 42,
    alpha: float = 0.05,
) -> tuple[pd.Series, pd.Series]:
    rng = np.random.default_rng(random_state)
    flt = flt_occ.to_numpy(dtype=np.float64)
    gnd = gnd_occ.to_numpy(dtype=np.float64)
    n_f, n_g = flt.shape[0], gnd.shape[0]
    n_p = flt.shape[1]

    deltas = np.empty((n_bootstrap, n_p), dtype=np.float64)
    for b in range(n_bootstrap):
        idx_f = rng.integers(0, n_f, size=n_f)
        idx_g = rng.integers(0, n_g, size=n_g)
        deltas[b] = flt[idx_f].mean(axis=0) - gnd[idx_g].mean(axis=0)

    low = np.percentile(deltas, 100.0 * (alpha / 2.0), axis=0)
    high = np.percentile(deltas, 100.0 * (1.0 - alpha / 2.0), axis=0)
    return (
        pd.Series(low, index=flt_occ.columns, name="bootstrap_ci_low"),
        pd.Series(high, index=flt_occ.columns, name="bootstrap_ci_high"),
    )


def permutation_test_delta(
    flt_occ: pd.DataFrame,
    gnd_occ: pd.DataFrame,
    observed_delta: pd.Series,
    n_perm: int = 1000,
    random_state: int = 42,
) -> pd.Series:
    rng = np.random.default_rng(random_state)
    flt = flt_occ.to_numpy(dtype=np.float64)
    gnd = gnd_occ.to_numpy(dtype=np.float64)
    n_f = flt.shape[0]
    combined = np.vstack([flt, gnd])
    n_total = combined.shape[0]

    obs_abs = np.abs(observed_delta.to_numpy(dtype=np.float64))
    exceed = np.zeros_like(obs_abs, dtype=np.int64)

    for _ in range(n_perm):
        perm_idx = rng.permutation(n_total)
        f_idx = perm_idx[:n_f]
        g_idx = perm_idx[n_f:]
        perm_delta = combined[f_idx].mean(axis=0) - combined[g_idx].mean(axis=0)
        exceed += (np.abs(perm_delta) >= obs_abs).astype(np.int64)

    pvals = exceed.astype(np.float64) / float(n_perm)
    return pd.Series(pvals, index=observed_delta.index, name="permutation_pvalue")


def compute_statmech_results(
    flt_sample_scores: pd.DataFrame,
    gnd_sample_scores: pd.DataFrame,
    accession: str,
    tissue: str,
    n_bootstrap: int = 1000,
    n_perm: int = 1000,
    eps: float = 1e-8,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    normalized_flt_scores = normalize_pathway_occupancies(flt_sample_scores)
    normalized_gnd_scores = normalize_pathway_occupancies(gnd_sample_scores)

    flight_mean_occupancy = normalized_flt_scores.mean(axis=0)
    ground_mean_occupancy = normalized_gnd_scores.mean(axis=0)
    delta_occupancy = flight_mean_occupancy - ground_mean_occupancy

    flight_energy = -np.log(normalized_flt_scores + eps)
    ground_energy = -np.log(normalized_gnd_scores + eps)
    flight_mean_energy = flight_energy.mean(axis=0)
    ground_mean_energy = ground_energy.mean(axis=0)
    delta_energy = flight_mean_energy - ground_mean_energy

    flt_ent = compute_entropy_per_sample(normalized_flt_scores, eps=eps)
    gnd_ent = compute_entropy_per_sample(normalized_gnd_scores, eps=eps)
    flight_mean_entropy = float(flt_ent["H"].mean())
    ground_mean_entropy = float(gnd_ent["H"].mean())
    delta_entropy = flight_mean_entropy - ground_mean_entropy

    ci_low, ci_high = bootstrap_delta_ci(
        normalized_flt_scores, normalized_gnd_scores, n_bootstrap=n_bootstrap, random_state=42
    )
    pvals = permutation_test_delta(
        normalized_flt_scores, normalized_gnd_scores, observed_delta=delta_occupancy, n_perm=n_perm, random_state=42
    )

    statmech_df = pd.DataFrame({
        "pathway": pathway_names,
        "flight_mean_occupancy": flight_mean_occupancy.reindex(pathway_names).to_numpy(),
        "ground_mean_occupancy": ground_mean_occupancy.reindex(pathway_names).to_numpy(),
        "delta_occupancy": delta_occupancy.reindex(pathway_names).to_numpy(),
        "flight_mean_energy": flight_mean_energy.reindex(pathway_names).to_numpy(),
        "ground_mean_energy": ground_mean_energy.reindex(pathway_names).to_numpy(),
        "delta_energy": delta_energy.reindex(pathway_names).to_numpy(),
        "bootstrap_ci_low": ci_low.reindex(pathway_names).to_numpy(),
        "bootstrap_ci_high": ci_high.reindex(pathway_names).to_numpy(),
        "permutation_pvalue": pvals.reindex(pathway_names).to_numpy(),
        "flight_mean_entropy": flight_mean_entropy,
        "ground_mean_entropy": ground_mean_entropy,
        "delta_entropy": delta_entropy,
        "n_flight": int(normalized_flt_scores.shape[0]),
        "n_ground": int(normalized_gnd_scores.shape[0]),
        "accession": accession,
        "tissue": tissue,
    })

    return statmech_df, normalized_flt_scores, normalized_gnd_scores


heartbeat(f"Prepared {total_groups} accession+tissue cohorts")

if "score_single_osdr_sample" not in globals() or not callable(score_single_osdr_sample):
    raise RuntimeError("score_single_osdr_sample is not available. Run Cell 11 before running this all-accession analysis cell.")

if len(eligible):
    probe_sample = str(eligible.iloc[0][sample_col])
    try:
        _raw_score_fn = score_single_osdr_sample
        _raw_params = inspect.signature(_raw_score_fn).parameters
        _needs_adapter = any(k not in _raw_params for k in ("use_disk_cache", "cache_dir", "return_cache_status"))

        if _needs_adapter:
            def score_single_osdr_sample(
                sample_name,
                top_k=10,
                use_disk_cache=False,
                cache_dir=None,
                return_cache_status=False,
                **kwargs,
            ):
                call_kwargs = {}
                if "top_k" in _raw_params:
                    call_kwargs["top_k"] = top_k
                for k, v in kwargs.items():
                    if k in _raw_params:
                        call_kwargs[k] = v

                out = _raw_score_fn(sample_name, **call_kwargs)

                if not return_cache_status:
                    return out

                if isinstance(out, tuple):
                    if len(out) == 6:
                        return out
                    if len(out) == 5:
                        sid, sample_vec, gene_level, sample_scores, sample_embedding = out
                        return sid, sample_vec, gene_level, sample_scores, False, sample_embedding
                    if len(out) == 4:
                        sid, sample_vec, gene_level, sample_scores = out
                        return sid, sample_vec, gene_level, sample_scores, False, np.array([], dtype=np.float32)

                raise ValueError(f"Unsupported score_single_osdr_sample return format: {type(out)}")

        _ = score_single_osdr_sample(probe_sample, top_k=10, use_disk_cache=True, cache_dir=disk_cache_dir)
    except Exception as e:
        raise RuntimeError(
            f"Preflight sample load failed for '{probe_sample}'. First error: {e}. Ensure Cell 11 ran and paths are valid."
        )

statmech_rows = []
cohort_summary = []
cache_index_rows = []

for cohort_idx, ((accession, tissue), g) in enumerate(groups, start=1):
    flt_samples = sorted(g.loc[g["condition_group"] == "flight", sample_col].unique().tolist())
    gnd_samples = sorted(g.loc[g["condition_group"] == "ground", sample_col].unique().tolist())

    if not flt_samples or not gnd_samples:
        continue

    heartbeat(
        f"Cohort {cohort_idx}/{total_groups}: {accession} | {tissue} | flight={len(flt_samples)} ground={len(gnd_samples)}"
    )

    flt_scores = []
    gnd_scores = []
    flt_genes = []
    gnd_genes = []

    for s_idx, s in enumerate(flt_samples, start=1):
        if s not in sample_score_cache:
            try:
                sid, _, gene_level_s, sample_scores_s, was_cached, sample_embedding_s = score_single_osdr_sample(
                    s, top_k=10, use_disk_cache=True, cache_dir=disk_cache_dir, return_cache_status=True
                )
                cache_stats["hit" if was_cached else "miss"] += 1
                sample_score_cache[s] = (sample_scores_s, gene_level_s, sid, sample_embedding_s)
            except Exception as e:
                reason = str(e)
                failed_samples.append({"sample": s, "accession": accession, "reason": reason})
                failed_reason_counts[reason] = failed_reason_counts.get(reason, 0) + 1
                continue
        sample_scores_s, gene_level_s, sid, _ = sample_score_cache[s]
        flt_scores.append(sample_scores_s)
        flt_genes.append(gene_level_s)
        if s_idx == 1 or s_idx == len(flt_samples) or s_idx % 5 == 0:
            heartbeat(f"  flight sample {s_idx}/{len(flt_samples)} scored: {sid}")

    for s_idx, s in enumerate(gnd_samples, start=1):
        if s not in sample_score_cache:
            try:
                sid, _, gene_level_s, sample_scores_s, was_cached, sample_embedding_s = score_single_osdr_sample(
                    s, top_k=10, use_disk_cache=True, cache_dir=disk_cache_dir, return_cache_status=True
                )
                cache_stats["hit" if was_cached else "miss"] += 1
                sample_score_cache[s] = (sample_scores_s, gene_level_s, sid, sample_embedding_s)
            except Exception as e:
                reason = str(e)
                failed_samples.append({"sample": s, "accession": accession, "reason": reason})
                failed_reason_counts[reason] = failed_reason_counts.get(reason, 0) + 1
                continue
        sample_scores_s, gene_level_s, sid, _ = sample_score_cache[s]
        gnd_scores.append(sample_scores_s)
        gnd_genes.append(gene_level_s)
        if s_idx == 1 or s_idx == len(gnd_samples) or s_idx % 5 == 0:
            heartbeat(f"  ground sample {s_idx}/{len(gnd_samples)} scored: {sid}")

    if not flt_scores or not gnd_scores:
        heartbeat(f"  skipped cohort {accession} | {tissue} because one condition had no usable samples")
        continue

    flt_sample_scores = sample_score_frames_to_matrix(flt_scores, value_col="mean_score")
    gnd_sample_scores = sample_score_frames_to_matrix(gnd_scores, value_col="mean_score")

    statmech_df, normalized_flt_scores, normalized_gnd_scores = compute_statmech_results(
        flt_sample_scores=flt_sample_scores,
        gnd_sample_scores=gnd_sample_scores,
        accession=accession,
        tissue=tissue,
        n_bootstrap=1000,
        n_perm=1000,
    )
    statmech_rows.append(statmech_df)

    normalized_scores_by_cohort[(accession, tissue)] = {
        "normalized_flt_scores": normalized_flt_scores,
        "normalized_gnd_scores": normalized_gnd_scores,
    }

    top_row = statmech_df.sort_values("delta_occupancy", ascending=False).iloc[0]
    top_pw = str(top_row["pathway"])
    flt_gene_mean = pd.concat(flt_genes).groupby(level=0).mean()
    gnd_gene_mean = pd.concat(gnd_genes).groupby(level=0).mean()
    members_top = Y_df.index[Y_df[top_pw] == 1]
    gene_delta = (flt_gene_mean.loc[members_top, top_pw] - gnd_gene_mean.loc[members_top, top_pw]).sort_values(ascending=False)
    top_driver_gene = str(gene_delta.index[0]) if len(gene_delta) else "NA"

    cohort_summary.append({
        "accession": accession,
        "tissue": tissue,
        "n_flight": int(top_row["n_flight"]),
        "n_ground": int(top_row["n_ground"]),
        "top_pathway_delta_occupancy": top_pw,
        "top_delta_occupancy": float(top_row["delta_occupancy"]),
        "top_pathway_delta_energy": str(statmech_df.sort_values("delta_energy").iloc[0]["pathway"]),
        "top_driver_gene_for_top_delta_occupancy": top_driver_gene,
        "delta_entropy": float(top_row["delta_entropy"]),
    })

    heartbeat(
        f"Finished cohort {cohort_idx}/{total_groups}: top occupancy shift = {top_pw} ({float(top_row['delta_occupancy']):.4f})"
    )

for sample_name in sorted(sample_score_cache.keys()):
    _, _, sample_id, sample_embedding = sample_score_cache[sample_name]
    sample_key = _cache_key(sample_name)
    cache_index_rows.append(
        {
            "sample_name": sample_name,
            "sample_id": sample_id,
            "cache_key": sample_key,
            "embedding_dim": int(len(sample_embedding)),
            "npz_path": str((disk_cache_dir / f"{sample_key}.npz").resolve()),
            "json_path": str((disk_cache_dir / f"{sample_key}.json").resolve()),
        }
    )

cache_index_df = pd.DataFrame(cache_index_rows)
failed_samples_df = pd.DataFrame(failed_samples)
all_accession_summary_df = pd.DataFrame(cohort_summary).sort_values(["top_delta_occupancy", "accession"], ascending=[False, True]) if cohort_summary else pd.DataFrame()

statmech_results_df = pd.concat(statmech_rows, ignore_index=True) if statmech_rows else pd.DataFrame()
all_accession_results_df = statmech_results_df.copy()

print(f"Eligible accession+tissue cohorts analyzed: {len(all_accession_summary_df)}")
print(f"Unique samples scored (in-memory cache): {len(sample_score_cache)}")
print(f"Disk cache hits: {cache_stats['hit']}")
print(f"Disk cache misses (new computations): {cache_stats['miss']}")
print(f"Failed sample loads: {len(failed_samples_df)}")
print(f"Disk cache directory: {disk_cache_dir}")
if failed_reason_counts:
    print("Top failure reasons:")
    for reason, n in sorted(failed_reason_counts.items(), key=lambda kv: kv[1], reverse=True)[:5]:
        print(f"- ({n}) {reason}")

if len(all_accession_summary_df):
    display(all_accession_summary_df.head(20))
else:
    print("No eligible cohorts found with both Space Flight and Ground Control samples.")

[heartbeat +0.0m] Prepared 84 accession+tissue cohorts
[heartbeat +0.0m] Cohort 1/84: OSD-100 | left eye | flight=6 ground=6
[heartbeat +0.0m]   flight sample 1/6 scored: OSD-100|Mmus_C57-6J_EYE_FLT_Rep1_M23
[heartbeat +0.2m]   flight sample 5/6 scored: OSD-100|Mmus_C57-6J_EYE_FLT_Rep5_M27
[heartbeat +0.3m]   flight sample 6/6 scored: OSD-100|Mmus_C57-6J_EYE_FLT_Rep6_M28
[heartbeat +0.3m]   ground sample 1/6 scored: OSD-100|Mmus_C57-6J_EYE_GC_Rep1_M33
[heartbeat +0.5m]   ground sample 5/6 scored: OSD-100|Mmus_C57-6J_EYE_GC_Rep5_M37
[heartbeat +0.6m]   ground sample 6/6 scored: OSD-100|Mmus_C57-6J_EYE_GC_Rep6_M38
[heartbeat +0.6m] Finished cohort 1/84: top occupancy shift = Interferon Alpha Response (0.0016)
[heartbeat +0.6m] Cohort 2/84: OSD-101 | Left gastrocnemius | flight=6 ground=6
[heartbeat +0.6m]   flight sample 1/6 scored: OSD-101|Mmus_C57-6J_GST_FLT_Rep1_M23
[heartbeat +0.8m]   flight sample 5/6 scored: OSD-101|Mmus_C57-6J_GST_FLT_Rep5_M27
[heartbeat +0.9m]   flight sample 6/6

,accession,tissue,n_flight,n_ground,top_pathway_delta_occupancy,top_delta_occupancy,top_pathway_delta_energy,top_driver_gene_for_top_delta_occupancy,delta_entropy
70,OSD-612,Left cerebral hemisphere,4,5,Interferon Alpha Response,0.004674,Mitotic Spindle,CXCL10,0.001208
35,OSD-397,Left retina,4,2,Myc Targets V2,0.004648,Myc Targets V2,GRWD1,0.004343
54,OSD-47,Liver,3,3,Interferon Alpha Response,0.003899,Apical Surface,DDX60,-0.006995
41,OSD-421,thymus,10,10,TNF-alpha Signaling via NF-kB,0.003395,Angiogenesis,NFIL3,-0.003869
21,OSD-244,Thymus,19,19,Interferon Alpha Response,0.003216,Bile Acid Metabolism,CD74,-0.004023
14,OSD-194,Left retina,5,3,Myc Targets V2,0.003070,Myc Targets V2,PPRC1,0.001264
29,OSD-270,Heart,3,2,Myc Targets V2,0.002977,Myc Targets V2,RRP12,-0.003985
13,OSD-173,liver,2,2,heme Metabolism,0.002938,Notch Signaling,MKRN1,-0.004612
19,OSD-242,Liver,5,4,Interferon Alpha Response,0.002756,Pperoxisome,EIF2AK2,-0.000802
45,OSD-457,Liver,12,12,Interferon Alpha Response,0.002569,Hedgehog Signaling,PSMB8,-0.004198


In [17]:
macrostate_summary_df = (
    statmech_results_df
    .sort_values(["accession", "tissue", "delta_occupancy"], ascending=[True, True, False])
    .groupby(["accession", "tissue"])
    .apply(lambda g: pd.Series({
        "n_flight": int(g["n_flight"].iloc[0]),
        "n_ground": int(g["n_ground"].iloc[0]),

        "top_positive_pathway": g.sort_values("delta_occupancy", ascending=False)["pathway"].iloc[0],
        "top_positive_delta": g["delta_occupancy"].max(),

        "top_negative_pathway": g.sort_values("delta_occupancy", ascending=True)["pathway"].iloc[0],
        "top_negative_delta": g["delta_occupancy"].min(),

        "mean_abs_delta_occupancy": g["delta_occupancy"].abs().mean(),
        "sum_abs_delta_occupancy": g["delta_occupancy"].abs().sum(),

        "flight_mean_entropy": g["flight_mean_entropy"].iloc[0],
        "ground_mean_entropy": g["ground_mean_entropy"].iloc[0],
        "delta_entropy": g["delta_entropy"].iloc[0],

        "n_significant_pathways": (
            (g["bootstrap_ci_low"] > 0) | (g["bootstrap_ci_high"] < 0)
        ).sum(),

        "top_significant_pathways": ", ".join(
            g.loc[
                (g["bootstrap_ci_low"] > 0) | (g["bootstrap_ci_high"] < 0)
            ]
            .sort_values("delta_occupancy", ascending=False)
            ["pathway"]
            .head(5)
            .tolist()
        )
    }))
    .reset_index()
)

display(macrostate_summary_df.head(20))

/tmp/ipykernel_2796/3058014310.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,accession,tissue,n_flight,n_ground,top_positive_pathway,top_positive_delta,top_negative_pathway,top_negative_delta,mean_abs_delta_occupancy,sum_abs_delta_occupancy,flight_mean_entropy,ground_mean_entropy,delta_entropy,n_significant_pathways,top_significant_pathways
0,OSD-100,left eye,6,6,Interferon Alpha Response,0.001591,Myc Targets V2,-0.001652,0.000311,0.015561,3.542461,3.544848,-0.002386,5,"IL-6/JAK/STAT3 Signaling, Allograft Rejection,..."
1,OSD-101,Left gastrocnemius,6,6,Myc Targets V1,0.000876,Myogenesis,-0.001462,0.000317,0.015844,3.572609,3.573166,-0.000558,6,"DNA Repair, PI3K/AKT/mTOR Signaling, Wnt-beta..."
2,OSD-102,Left kidney,6,6,TNF-alpha Signaling via NF-kB,0.001614,Interferon Alpha Response,-0.001926,0.000360,0.018006,3.562062,3.563783,-0.001721,7,"TNF-alpha Signaling via NF-kB, Mitotic Spindle..."
3,OSD-103,Left quadriceps femoris,6,6,Cholesterol Homeostasis,0.001926,Adipogenesis,-0.000820,0.000342,0.017093,3.575241,3.571968,0.003273,8,"Unfolded Protein Response, Myogenesis, Mitotic..."
4,OSD-104,Soleus-both sides,6,6,Adipogenesis,0.001244,TNF-alpha Signaling via NF-kB,-0.002561,0.000431,0.021549,3.580676,3.581573,-0.000897,10,"Adipogenesis, Oxidative Phosphorylation, Ppero..."
5,OSD-105,Left tibialis anterior,6,6,Myc Targets V2,0.001794,Oxidative Phosphorylation,-0.001779,0.000396,0.019788,3.577340,3.574769,0.002571,5,"Myc Targets V2, TGF-beta Signaling, IL-2/STAT5..."
6,OSD-137,Liver,6,6,Fatty Acid Metabolism,0.000854,Cholesterol Homeostasis,-0.001604,0.000341,0.017073,3.555445,3.553987,0.001458,9,"Fatty Acid Metabolism, p53 Pathway, KRAS Signa..."
7,OSD-161,Adrenal gland,5,5,Myc Targets V2,0.001217,TNF-alpha Signaling via NF-kB,-0.001364,0.000230,0.011517,3.573647,3.571854,0.001793,5,"Bile Acid Metabolism, DNA Repair, UV Response ..."
8,OSD-162,eye,5,5,Cholesterol Homeostasis,0.002137,Myc Targets V2,-0.002975,0.000494,0.024705,3.547970,3.545210,0.002760,9,"Cholesterol Homeostasis, Pperoxisome, Angiogen..."
9,OSD-163,Left kidney,6,6,G2-M Checkpoint,0.001345,Angiogenesis,-0.001241,0.000332,0.016579,3.559054,3.562455,-0.003401,6,"G2-M Checkpoint, Allograft Rejection, KRAS Sig..."


In [14]:
# -------------------------
# Save all-accession condition analysis reports
# -------------------------
report_dir = CFG["out_dir"] / "all_accessions"
report_dir.mkdir(parents=True, exist_ok=True)

detail_csv = report_dir / "all_accession_condition_pathway_scores.csv"
statmech_csv = report_dir / "all_accession_statmech_results.csv"
summary_csv = report_dir / "all_accession_condition_summary.csv"
failed_csv = report_dir / "all_accession_failed_samples.csv"
summary_md = report_dir / "all_accession_condition_summary.md"
cache_index_csv = report_dir / "osdr_embedding_cache_index.csv"

if "statmech_results_df" not in globals() or statmech_results_df.empty:
    print("No statmech results found. Run the previous cell first.")
else:
    # Keep legacy filename and save explicit statmech filename.
    statmech_results_df.to_csv(detail_csv, index=False)
    statmech_results_df.to_csv(statmech_csv, index=False)
    all_accession_summary_df.to_csv(summary_csv, index=False)
    failed_samples_df.to_csv(failed_csv, index=False)
    if "cache_index_df" in globals() and not cache_index_df.empty:
        cache_index_df.to_csv(cache_index_csv, index=False)

    # Human-readable markdown summary
    with open(summary_md, "w", encoding="utf-8") as f:
        f.write("# All-accession pathway condition analysis\n\n")
        f.write("Condition contrast: Space Flight minus Ground Control\n\n")
        f.write("Stat-mech fields saved per pathway: occupancy, pseudo-energy, entropy, bootstrap CI, permutation p-value\n\n")
        f.write(f"Cohorts analyzed: {len(all_accession_summary_df)}\n\n")
        f.write(f"Total pathway rows: {len(statmech_results_df)}\n\n")
        f.write(f"Unique samples scored: {len(sample_score_cache)}\n\n")
        if "cache_stats" in globals():
            f.write(f"Disk cache hits: {cache_stats.get('hit', 0)}\n\n")
            f.write(f"Disk cache misses: {cache_stats.get('miss', 0)}\n\n")
        f.write("## Top pathway per accession+tissue cohort\n\n")
        f.write(all_accession_summary_df.to_markdown(index=False))
        if len(failed_samples_df):
            f.write("\n\n## Failed samples\n\n")
            f.write(failed_samples_df.to_markdown(index=False))

    print("Saved all-accession reports:")
    print("-", detail_csv)
    print("-", statmech_csv)
    print("-", summary_csv)
    print("-", failed_csv)
    print("-", summary_md)
    if "cache_index_df" in globals() and not cache_index_df.empty:
        print("-", cache_index_csv)

Saved all-accession reports:
- results/pathway_head/all_accessions/all_accession_condition_pathway_scores.csv
- results/pathway_head/all_accessions/all_accession_statmech_results.csv
- results/pathway_head/all_accessions/all_accession_condition_summary.csv
- results/pathway_head/all_accessions/all_accession_failed_samples.csv
- results/pathway_head/all_accessions/all_accession_condition_summary.md
- results/pathway_head/all_accessions/osdr_embedding_cache_index.csv


KeyError: 'n_significant_pathways'